In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/datasets/liamguske/ground-truth/test2_new.txt
/kaggle/input/datasets/liamguske/music-recommender/sample_submission.csv
/kaggle/input/datasets/liamguske/music-recommender/testItem2.txt
/kaggle/input/datasets/liamguske/music-recommender/albumData2.txt
/kaggle/input/datasets/liamguske/music-recommender/artistData2.txt
/kaggle/input/datasets/liamguske/music-recommender/trackData2.txt
/kaggle/input/datasets/liamguske/music-recommender/trainItem2.txt
/kaggle/input/datasets/liamguske/music-recommender/genreData2.txt
/kaggle/input/competitions/music-recommender-2026s/sample_submission.csv
/kaggle/input/competitions/music-recommender-2026s/testItem2.txt
/kaggle/input/competitions/music-recommender-2026s/albumData2.txt
/kaggle/input/competitions/music-recommender-2026s/artistData2.txt
/kaggle/input/competitions/music-recommender-2026s/trackData2.txt
/kaggle/input/competitions/music-recommender-2026s/trainItem2.txt
/kaggle/input/competitions/music-recommender-2026s/genreData2.txt


In [2]:
directory = '/kaggle/input/datasets/liamguske/music-recommender/'
trainData = os.path.join(directory, 'trainItem2.txt')
testData = os.path.join(directory, 'testItem2.txt')
trackData = os.path.join(directory, 'trackData2.txt')
albumData = os.path.join(directory, 'albumData2.txt')
artistData = os.path.join(directory, 'artistData2.txt')
genreData = os.path.join(directory, 'genreData2.txt')

Check File Existence

In [3]:
trainFile = os.path.exists(trainData)
testFile = os.path.exists(testData)
trackFile = os.path.exists(trackData)
albumFile = os.path.exists(albumData)
artistFile = os.path.exists(artistData)
genreFile = os.path.exists(genreData)

In [4]:
def file_processing():
    if not testFile:
        print(f"{trainData} does not exist. Please check it")
    elif not trainFile:
        print(f"{testData} does not exist. Please check it")
    elif not trackFile:
        print(f"{trackData} does not exist. Please check it")
    elif not albumFile:
        print(f"{albumData} does not exist. Please check it")
    elif not artistFile:
        print(f"{artistData} does not exist. Please check it")
    elif not genreFile:
        print(f"{genreData} does not exist. Please check it")
    else:
        print("All files properly uploaded.")

file_processing()

All files properly uploaded.


Feature Engineering
Track: creates Dictionary with album, artist, and genres
Album: creates Dictionary with artist and genres
Artist: creates Set with all artists
Genres: creates Set with all genres

In [5]:
trackInfo = {}
albumInfo = {}
artistSet = set()
genreSet = set()
print("Loading Features...\n")
with open(trackData, "r") as myTracks:
    for line in myTracks:
        element = line.rstrip().split('|')
        if len(element) < 3:
            continue
        tid = element[0]
        trackInfo[tid] = {
            "album": element[1],
            "artist": element[2],
            "genres": element[3:] if len(element) > 3 else []
        }

with open(albumData, "r") as myAlbums:
    for line in myAlbums:
        element = line.rstrip().split("|")
        if len(element) < 2:
            continue
        aid = element[0]
        albumInfo[aid] = {
            "artist": element[1],
            "genres": element[2:] if len(element) > 2 else []
        }

with open(artistData, "r") as myArtists:
    for line in myArtists:
        curr = line.rstrip()
        if curr:
            artistSet.add(curr)

with open(genreData, "r") as myGenres:
    for line in myGenres:
        curr = line.rstrip()
        if curr:
            genreSet.add(curr)
print(f"Total Features Loaded:\n {len(trackInfo)} tracks\n {len(albumInfo)} albums\n {len(artistSet)} artists\n {len(genreSet)} genres\n")

Loading Features...

Total Features Loaded:
 224041 tracks
 52829 albums
 18674 artists
 567 genres



Creates dictionaries for album -> tracks and artists -> albums
Ensuring tracks are properly assigned to correct albums and albums are assigned to correct artist

Creates dictionary for genre -> track
Ensuring tracks are properly assigned to correct genre

In [6]:
albumToTracks = {}
artistsToAlbums = {}

for tid, x in trackInfo.items():
    album = x["album"]
    artist = x["artist"]
    albumToTracks.setdefault(album, []).append(tid)
    artistsToAlbums.setdefault(artist, set()).add(album)

genreToTracks = {}
for tid, x in trackInfo.items():
    for i in x["genres"]:
        genreToTracks.setdefault(i, []).append(tid)

print(f"Reverse Mapping Generated\nalbum -> tracks\nartist -> album\ngenre -> track")

Reverse Mapping Generated
album -> tracks
artist -> album
genre -> track


Function to classify the key as the correct category (track, album, artist, or genre)

In [7]:
print(f"Function to classify training key as either track, album, artist, or genre")
def classifier(key):
    if key in trackInfo: return "track"
    if key in albumInfo: return "album"
    if key in artistSet: return "artist"
    if key in genreSet: return "genre"
    return "N/A"

Function to classify training key as either track, album, artist, or genre


Bulds dictionaries for each user and the statistics they have provided on the track hierarchy


In [8]:
train = {}
test = {}

with open(trainData, "r") as trainRec:
    curr_user = None
    for line in trainRec:
        line = line.strip()
        if not line:
            continue
        if '|' in line:
            curr_user = line.split('|')[0]
            train[curr_user] = {"track": {}, "album": {}, "artist": {}, "genre": {}}
        elif '\t' in line and curr_user is not None:
            key, rating = line.split('\t', 1)
            rating = float(rating)
            cls = classifier(key)
            if cls != "N/A":
                train[curr_user][cls][key] = rating

print(f"{len(train)} training users loaded")
for uid, profile in list(train.items())[:3]:
    counts = {k: len(v) for k, v in profile.items()}
    print(f"  {uid}: {counts}")
with open(testData, "r") as testRec:
    curr_user = None
    for line in testRec:
        line = line.strip()
        if not line:
            continue
        if '|' in line:
            curr_user = line.split('|')[0]
            test[curr_user] = []
        elif curr_user is not None:
            test[curr_user].append(line)
print(f"{len(test)} test users loaded")

49204 training users loaded
  199808: {'track': 0, 'album': 0, 'artist': 31, 'genre': 4}
  199809: {'track': 0, 'album': 0, 'artist': 32, 'genre': 6}
  199810: {'track': 58, 'album': 3, 'artist': 74, 'genre': 5}
20000 test users loaded


Creates global average

In [9]:
allRatings = [a for b in train.values() for a in b["track"].values()]
globalAvg = np.mean(allRatings)
print(f"Global average rating: {globalAvg:.3f}")

Global average rating: 49.854


In [10]:
# Global popularity: aggregate ratings across ALL users per entity
albumGlobalAvg = {}
artistGlobalAvg = {}
genreGlobalAvg = {}
trackGlobalAvg = {}

for uid, profile in train.items():
    for aid, rating in profile["album"].items():
        albumGlobalAvg.setdefault(aid, []).append(rating)
    for aid, rating in profile["artist"].items():
        artistGlobalAvg.setdefault(aid, []).append(rating)
    for gid, rating in profile["genre"].items():
        genreGlobalAvg.setdefault(gid, []).append(rating)
    for tid, rating in profile["track"].items():
        trackGlobalAvg.setdefault(tid, []).append(rating)

albumGlobalAvg = {k: np.mean(v) for k, v in albumGlobalAvg.items()}
artistGlobalAvg = {k: np.mean(v) for k, v in artistGlobalAvg.items()}
genreGlobalAvg  = {k: np.mean(v) for k, v in genreGlobalAvg.items()}
trackGlobalAvg  = {k: np.mean(v) for k, v in trackGlobalAvg.items()}

In [11]:
def isColdStart(userProf):
    return all(len(v) == 0 for v in userProf.values())

def get_track_score(userProf, tid):
    # Primary: direct track rating
    if tid in userProf["track"]:
        return userProf["track"][tid]
    
    # Secondary: mean of sibling tracks in same album
    if tid in trackInfo:
        album_id = trackInfo[tid]["album"]
        rated_tracks = [
            userProf["track"][t]
            for t in albumToTracks.get(album_id, [])
            if t in userProf["track"] and t != tid
        ]
        if rated_tracks:
            return np.mean(rated_tracks)
    
    return None  # no track signal found

def get_album_score(userProf, album_id):
    # Primary: direct album rating
    if album_id in userProf["album"]:
        return userProf["album"][album_id]
    
    # Secondary (Dig Deeper): mean track rating for this album
    rated_tracks = [
        userProf["track"][tid]
        for tid in albumToTracks.get(album_id, [])
        if tid in userProf["track"]
    ]
    if rated_tracks:
        return np.mean(rated_tracks)
    
    # Tertiary: global album average
    return albumGlobalAvg.get(album_id, globalAvg)

In [12]:
def collectSignals(userProf, tid):
    if tid not in trackInfo:
        return None

    x = trackInfo[tid]

    # Cold start: no user history at all, use global signals only
    if isColdStart(userProf):
        album_score  = albumGlobalAvg.get(x["album"], globalAvg)
        artist_score = artistGlobalAvg.get(x["artist"], globalAvg)
        genre_scores = [genreGlobalAvg.get(g, globalAvg) for g in x["genres"]]
    else:
        # Hierarchical: track -> album -> artist -> genre
        track_score = get_track_score(userProf, tid)

        # Album score with dig deeper
        album_score = get_album_score(userProf, x["album"])

        # Artist score direct only
        artist_score = userProf["artist"].get(x["artist"], globalAvg)

        # Genre scores direct only, exclude unseen
        genre_scores = [userProf["genre"][g] for g in x["genres"] if g in userProf["genre"]]

    if genre_scores:
        g_count        = len(genre_scores)
        g_max          = np.max(genre_scores)
        g_min          = np.min(genre_scores)
        g_mean         = np.mean(genre_scores)
        g_median       = np.median(genre_scores)
        g_std          = np.std(genre_scores)
        g_variance     = np.var(genre_scores)
        g_range        = np.ptp(genre_scores)
        g_percentile75 = np.percentile(genre_scores, 75)
        g_top2Mean     = np.mean(np.sort(genre_scores)[-2:])
        g_coverage     = len(genre_scores) / len(x["genres"]) if x["genres"] else 0
    else:
        g_count        = 0
        g_max          = globalAvg
        g_min          = globalAvg
        g_mean         = globalAvg
        g_median       = globalAvg
        g_std          = 0
        g_variance     = 0
        g_range        = 0
        g_percentile75 = globalAvg
        g_top2Mean     = globalAvg
        g_coverage     = 0

    result = {
        "Album Score":              album_score,
        "Artist Score":             artist_score,
        "Genre Count":              g_count,
        "Genre Max":                g_max,
        "Genre Min":                g_min,
        "Genre Mean":               g_mean,
        "Genre Median":             g_median,
        "Genre Standard Deviation": g_std,
        "Genre Variance":           g_variance,
        "Genre Range":              g_range,
        "Genre 75th Percentile":    g_percentile75,
        "Genre Top 2 Mean":         g_top2Mean,
        "Genre Coverage Ratio":     g_coverage
    }

    # If direct track signal exists, include it
    if not isColdStart(userProf):
        result["Track Score"] = track_score if track_score is not None else globalAvg

    return result

In [13]:
def run_predictions(score_fn):
    preds = {}
    for uid, track_list in test.items():
        profile = train.get(uid, {"track": {}, "album": {}, "artist": {}, "genre": {}})

        scores = {}
        preds[uid] = {}
        
        for tid in track_list:
            scores[tid] = score_fn(profile, tid)

        sortedTracks = sorted(scores, key=lambda t: scores[t], reverse=True)
        for i, tid in enumerate(sortedTracks):
            preds[uid][tid] = 1 if i < 3 else 0

    return preds

In [14]:
def save_submission(predictions, method):
    path = f'/kaggle/working/submission.csv'
    with open(path, 'w') as f:
        f.write("TrackID,Predictor\n")
        for uid, tracks in predictions.items():
            for tid, pred in tracks.items():
                f.write(f"{uid}_{tid},{int(pred)}\n")

    # quick sanity count
    total = sum(len(t) for t in predictions.values())
    likes = sum(p for t in predictions.values() for p in t.values())
    print(f"\n── {method} ──────────────────────────────────────")
    print(f"  Rows     : {total}")
    print(f"  Likes    : {likes}  ({likes/total*100:.1f}%)")
    print(f"  Dislikes : {total-likes}  ({(total-likes)/total*100:.1f}%)")
    print(f"  Saved to : {path}")

In [15]:

 
# ─── Step 1: Parse test2_new.txt ground truth ────────────────────────────────
groundTruth = "/kaggle/input/datasets/liamguske/ground-truth/test2_new.txt"   # ← adjust path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, StringType, DoubleType
from pyspark.sql.functions import udf
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
 
spark = SparkSession.builder.appName("MusicRecGBT_CV").getOrCreate()
 
# ─── Step 1: Parse test2_new.txt ground truth ────────────────────────────────
gt_rows = []
with open(groundTruth, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('|')
        if len(parts) != 3:
            continue
        uid, tid, label = parts[0], parts[1], int(parts[2])
        gt_rows.append((uid, tid, label))
 
print(f"Ground-truth rows : {len(gt_rows)}")
print(f"Unique users      : {len(set(r[0] for r in gt_rows))}")
print(f"Likes / Dislikes  : {sum(r[2] for r in gt_rows)} / {len(gt_rows) - sum(r[2] for r in gt_rows)}")
 
# ─── Step 2: Feature builder for (user, track) pairs — 14 features ───────────
FEATURE_NAMES = [
    "track_score", "album_score", "artist_score",
    "g_count", "g_max", "g_min", "g_mean", "g_median",
    "g_std", "g_var", "g_range", "g_p75", "g_top2", "g_coverage",
]
 
def build_features(userProf, tid):
    if tid not in trackInfo:
        # Unknown track — globals across the board, zeros for dispersion / count
        return [globalAvg, globalAvg, globalAvg,
                0.0, globalAvg, globalAvg, globalAvg, globalAvg,
                0.0, 0.0, 0.0, globalAvg, globalAvg, 0.0]
 
    info = trackInfo[tid]
 
    # Track score: direct → sibling-album mean → global track mean → globalAvg
    if tid in userProf["track"]:
        track_score = userProf["track"][tid]
    else:
        siblings = [
            userProf["track"][t]
            for t in albumToTracks.get(info["album"], [])
            if t in userProf["track"] and t != tid
        ]
        track_score = float(np.mean(siblings)) if siblings else trackGlobalAvg.get(tid, globalAvg)
 
    album_score  = get_album_score(userProf, info["album"])
    artist_score = userProf["artist"].get(info["artist"],
                       artistGlobalAvg.get(info["artist"], globalAvg))
 
    # All 11 genre statistics (same definitions as original collectSignals)
    ratings = [userProf["genre"][g] for g in info["genres"] if g in userProf["genre"]]
    if ratings:
        g_count    = float(len(ratings))
        g_max      = float(np.max(ratings))
        g_min      = float(np.min(ratings))
        g_mean     = float(np.mean(ratings))
        g_median   = float(np.median(ratings))
        g_std      = float(np.std(ratings))
        g_var      = float(np.var(ratings))
        g_range    = float(np.ptp(ratings))
        g_p75      = float(np.percentile(ratings, 75))
        g_top2     = float(np.mean(np.sort(ratings)[-2:]))
        g_coverage = float(len(ratings) / len(info["genres"])) if info["genres"] else 0.0
    else:
        g_count    = 0.0
        g_max      = globalAvg
        g_min      = globalAvg
        g_mean     = globalAvg
        g_median   = globalAvg
        g_std      = 0.0
        g_var      = 0.0
        g_range    = 0.0
        g_p75      = globalAvg
        g_top2     = globalAvg
        g_coverage = 0.0
 
    return [float(x) for x in (
        track_score, album_score, artist_score,
        g_count, g_max, g_min, g_mean, g_median,
        g_std, g_var, g_range, g_p75, g_top2, g_coverage,
    )]
 
# ─── Step 3: Build training DataFrame from ground truth ──────────────────────
empty_profile = {"track": {}, "album": {}, "artist": {}, "genre": {}}
training_rows = []
for uid, tid, label in gt_rows:
    profile = train.get(uid, empty_profile)
    feats = build_features(profile, tid)
    training_rows.append((uid, tid, *feats, int(label)))
 
schema = StructType([
    StructField("userID",  StringType()),
    StructField("trackID", StringType()),
    *[StructField(n, FloatType()) for n in FEATURE_NAMES],
    StructField("label",   IntegerType()),
])
trainDF = spark.createDataFrame(training_rows, schema)
print(f"\nTraining DataFrame: {trainDF.count()} rows × {len(FEATURE_NAMES)} features")
trainDF.show(5)
 
# ─── Step 4: Assemble features ───────────────────────────────────────────────
assembler = VectorAssembler(inputCols=FEATURE_NAMES, outputCol="features")
trainAssembled = assembler.transform(trainDF).cache()
trainAssembled.count()  # materialise for CV
 
# ─── Step 5: 3-fold CV grid search ───────────────────────────────────────────
gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    subsamplingRate=0.8, seed=42)
 
paramGrid = (ParamGridBuilder()
             .addGrid(gbt.maxDepth, [3, 5, 7])
             .addGrid(gbt.maxIter,  [50, 100])
             .addGrid(gbt.stepSize, [0.05, 0.1])
             .build())
 
evaluator = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
 
cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)
 
print(f"\nRunning 3-fold CV over {len(paramGrid)} param combos "
      f"({3 * len(paramGrid)} fits, parallelism=2)...")
cvModel   = cv.fit(trainAssembled)
bestModel = cvModel.bestModel  # auto-refit on full trainAssembled by CV default
 
print(f"\nBest CV AUC: {max(cvModel.avgMetrics):.4f}")
print(f"Best hyperparameters:")
print(f"  maxDepth = {bestModel.getMaxDepth()}")
print(f"  maxIter  = {bestModel.getMaxIter()}")
print(f"  stepSize = {bestModel.getStepSize()}")
 
# Top-3 param combinations for context
print("\nTop 3 param combinations by avg CV AUC:")
ranked = sorted(zip(cvModel.getEstimatorParamMaps(), cvModel.avgMetrics),
                key=lambda x: x[1], reverse=True)
for params, auc in ranked[:3]:
    p = {k.name: v for k, v in params.items()}
    print(f"  AUC={auc:.4f}  {p}")
 
# ─── Step 6: Build feature DataFrame for the actual Kaggle test set ──────────
test_rows = []
for uid, track_list in test.items():
    profile = train.get(uid, empty_profile)
    for tid in track_list:
        feats = build_features(profile, tid)
        test_rows.append((uid, tid, *feats))
 
test_schema = StructType([
    StructField("userID",  StringType()),
    StructField("trackID", StringType()),
    *[StructField(n, FloatType()) for n in FEATURE_NAMES],
])
testDF        = spark.createDataFrame(test_rows, test_schema)
testAssembled = assembler.transform(testDF)
 
# ─── Step 7: Predict probabilities ───────────────────────────────────────────
testPreds = bestModel.transform(testAssembled)
 
prob_udf  = udf(lambda v: float(v[1]), DoubleType())
testPreds = testPreds.withColumn("p1", prob_udf("probability"))
prob_rows = testPreds.select("userID", "trackID", "p1").collect()
 
# ─── Step 8: Rank top-3 per user → 1, rest → 0 ───────────────────────────────
user_scores = {}
for row in prob_rows:
    user_scores.setdefault(row.userID, {})[row.trackID] = row.p1
 
preds = {}
for uid, track_list in test.items():
    scores = user_scores.get(uid, {})
    full   = {tid: scores.get(tid, globalAvg) for tid in track_list}
    ranked_tracks = sorted(full, key=lambda t: full[t], reverse=True)
    preds[uid] = {tid: (1 if i < 3 else 0) for i, tid in enumerate(ranked_tracks)}
 
best_auc = max(cvModel.avgMetrics)
save_submission(preds, f'GBT + CV (best AUC={best_auc:.4f}, '
                       f'depth={bestModel.getMaxDepth()}, '
                       f'iter={bestModel.getMaxIter()}, '
                       f'step={bestModel.getStepSize()})')
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 20:00:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ground-truth rows : 6000
Unique users      : 1000
Likes / Dislikes  : 3000 / 3000



Training DataFrame: 6000 rows × 14 features
+------+-------+-----------+-----------+------------+-------+---------+---------+---------+---------+-----+-----+-------+---------+---------+----------+-----+
|userID|trackID|track_score|album_score|artist_score|g_count|    g_max|    g_min|   g_mean| g_median|g_std|g_var|g_range|    g_p75|   g_top2|g_coverage|label|
+------+-------+-----------+-----------+------------+-------+---------+---------+---------+---------+-----+-----+-------+---------+---------+----------+-----+
|200031|  30877|  53.426647|       90.0|        50.0|    2.0|     90.0|     80.0|     85.0|     85.0|  5.0| 25.0|   10.0|     87.5|     85.0|0.33333334|    1|
|200031|   8244|  55.757526|       90.0|   37.971428|    2.0|     90.0|     80.0|     85.0|     85.0|  5.0| 25.0|   10.0|     87.5|     85.0|       0.4|    1|
|200031| 130183|   67.61905|   67.61905|   49.853992|    0.0|49.853992|49.853992|49.853992|49.853992|  0.0|  0.0|    0.0|49.853992|49.853992|       0.0|    0|
|

26/05/08 20:00:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



Running 3-fold CV over 12 param combos (36 fits, parallelism=2)...


26/05/08 20:00:52 WARN BlockManager: Block rdd_28_0 already exists on this machine; not re-adding it
26/05/08 20:04:32 WARN DAGScheduler: Broadcasting large task binary with size 1001.1 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1002.8 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1004.1 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1004.6 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1011.4 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1005.6 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1006.4 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1008.7 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1013.4 KiB
26/05/08 20:04:33 WARN DAGScheduler: Broadcasting large task binary with size 1022.0 KiB
26/05/08 


Best CV AUC: 0.9143
Best hyperparameters:
  maxDepth = 3
  maxIter  = 50
  stepSize = 0.1

Top 3 param combinations by avg CV AUC:
  AUC=0.9143  {'maxDepth': 3, 'maxIter': 50, 'stepSize': 0.1}
  AUC=0.9139  {'maxDepth': 3, 'maxIter': 100, 'stepSize': 0.05}
  AUC=0.9129  {'maxDepth': 3, 'maxIter': 100, 'stepSize': 0.1}


26/05/08 20:19:21 WARN TaskSetManager: Stage 27852 contains a task of very large size (4157 KiB). The maximum recommended task size is 1000 KiB.



── GBT + CV (best AUC=0.9143, depth=3, iter=50, step=0.1) ──────────────────────────────────────
  Rows     : 120000
  Likes    : 60000  (50.0%)
  Dislikes : 60000  (50.0%)
  Saved to : /kaggle/working/submission.csv
